# 02 — Data Preparation

Fresh-kernel, handoff-driven continuous-regression preparation for UCI Dataset 165. No model is trained or evaluated.

## 1. Preparation Context and Handoff Boundary

The persisted Notebook-01 handoff is the sole upstream scientific authority; no live Notebook-01 state is reused.

In [1]:
from __future__ import annotations
import json
from pathlib import Path
import pandas as pd
def display(value):
    print(value)
from scripts.build_exploration_handoff import load_and_validate_exploration_handoff
from scripts.download_data import acquire_uci_dataset
from scripts.prepare_data import (ContinuousRegressionSplitPolicy, analyze_repeated_profiles_across_partitions, build_feature_manifest, build_preparation_handoff_manifest, build_preparation_manifest, build_quality_evidence, build_split_manifest, fingerprint_dataframe, fingerprint_dataframe_csv, fingerprint_file, load_and_validate_preparation_handoff, prepare_tabular_dataset, separate_dataset_roles, split_continuous_regression_dataset, validate_prepared_dataset, validate_raw_dataset, validate_regression_partitions, validate_source_against_exploration_handoff, write_preparation_artifacts)
from scripts.project_context import get_project_context
PROJECT=get_project_context(); DATASET_SLUG="concrete-compressive-strength"; UCI_DATASET_ID=165
EXPLORATION_HANDOFF_PATH=Path("artifacts/exploration")/DATASET_SLUG/"exploration-handoff.json"

## 2. Independent Exploration-Handoff Loading

Contracts and readiness are reconstructed from disk. Continuous regression has no artificial classes.

In [2]:
exploration_handoff_file=PROJECT.require_file(EXPLORATION_HANDOFF_PATH)
h=load_and_validate_exploration_handoff(exploration_handoff_file,expected_dataset_slug=DATASET_SLUG,expected_source_dataset_id=UCI_DATASET_ID)
s=h["source"]; pc=h["prediction_contract"]; fc=h["feature_contract"]; sc=h["preparation_contract"]["split_policy"]
TARGET_COLUMN=pc["target_column"]; TARGET_CLASSES=tuple(pc["target_classes"]); FEATURE_COLUMNS=tuple(fc["feature_columns"]); NUMERICAL_FEATURES=tuple(fc["numerical_features"]); CATEGORICAL_FEATURES=tuple(fc["categorical_features"]); IDENTIFIER_COLUMNS=tuple(fc["identifier_columns"])
assert pc["problem_type"]=="continuous_regression" and TARGET_CLASSES==() and pc["positive_class"] is None and pc["class_semantics"] is None
assert pc["target_semantics"]=="Continuous / quantitative" and pc["target_unit"]=="MPa"
assert h["readiness"]["deterministic_preparation_ready"] and h["readiness"]["split_execution_ready"]
EXPLORATION_SHA256=fingerprint_file(exploration_handoff_file)
assert EXPLORATION_SHA256=="91f9e9349c81db1aa041e307898c6811f2e5c421c99a64e6637dc1db49465387"

## 3. Independent UCI Source Acquisition

Dataset 165 is acquired through the project helper and read into a new dataframe.

In [3]:
acquisition=acquire_uci_dataset(dataset_id=int(s["dataset_id"]),destination=Path("data/raw")/DATASET_SLUG,project_root=PROJECT.root)
DATASET_FILE=acquisition.require_one_file("dataset.csv"); METADATA_FILE=acquisition.require_one_file("metadata.json"); VARIABLES_FILE=acquisition.require_one_file("variables.csv")
raw_df=pd.read_csv(DATASET_FILE); print(raw_df.shape)

(1030, 9)


## 4. Source Identity and Continuous Regression Contract Revalidation

The fail-closed gate authenticates bytes, schema, UCI roles, continuous target validity, and the slag metadata-resolution evidence.

In [4]:
source_identity=validate_source_against_exploration_handoff(raw_df,handoff=h,source_file=DATASET_FILE,project_root=PROJECT.root,dataset_slug=DATASET_SLUG,source_repository=s["repository"],source_dataset_id=UCI_DATASET_ID,metadata_file=METADATA_FILE,variables_file=VARIABLES_FILE)
COLUMN_ORDER=source_identity.column_order; EXPECTED_DTYPES={c:"numeric" for c in COLUMN_ORDER}
raw_validation=validate_raw_dataset(raw_df,column_order=COLUMN_ORDER,identifier_columns=IDENTIFIER_COLUMNS,feature_columns=FEATURE_COLUMNS,target_column=TARGET_COLUMN,target_classes=(),categorical_expected_values={},expected_types=EXPECTED_DTYPES,problem_type="continuous_regression")
RAW_SNAPSHOT=raw_df.copy(deep=True); RAW_LOGICAL_FINGERPRINT=fingerprint_dataframe(raw_df); SOURCE_SHA256=fingerprint_file(DATASET_FILE)
assert SOURCE_SHA256==s["sha256"]

## 5. Defensive Prepared Projection

Only a deep defensive copy is authorized: zero rounding, truncation, imputation, clipping, outlier removal, deduplication, or target transformation.

In [5]:
prepared_result=prepare_tabular_dataset(raw_df); prepared_df=prepared_result.dataframe
prepared_validation=validate_prepared_dataset(raw_df,prepared_df,column_order=COLUMN_ORDER,identifier_columns=IDENTIFIER_COLUMNS,feature_columns=FEATURE_COLUMNS,target_column=TARGET_COLUMN,target_classes=(),categorical_expected_values={},expected_types=EXPECTED_DTYPES,authorized_changed_columns=(),expected_row_count=1030,expected_materialized_counts={},observed_materialized_counts=dict(prepared_result.materialized_counts),problem_type="continuous_regression")
pd.testing.assert_frame_equal(raw_df,RAW_SNAPSHOT); pd.testing.assert_frame_equal(prepared_df,RAW_SNAPSHOT)
DECIMAL_SLAG_ROWS=int(((prepared_df["Blast Furnace Slag"]%1).abs()>0).sum()); assert DECIMAL_SLAG_ROWS==298
assert prepared_df is not raw_df and prepared_result.rules==() and fingerprint_file(DATASET_FILE)==SOURCE_SHA256

## 6. Feature and Target Separation

X contains the eight ordered numeric predictors; y remains continuous in MPa. No encoder or membership token is created.

In [6]:
roles=separate_dataset_roles(prepared_df,identifier_columns=IDENTIFIER_COLUMNS,feature_columns=FEATURE_COLUMNS,target_column=TARGET_COLUMN)
lineage,X,y=roles.lineage,roles.features,roles.target
assert lineage.shape==(1030,0) and tuple(X.columns)==FEATURE_COLUMNS and TARGET_COLUMN not in X and pd.api.types.is_numeric_dtype(y)

## 7. Split Policy Reconstruction

The handoff freezes 70/15/15, seed 42, shuffle true, stratify null, and an educational static snapshot with operational validity unconfirmed.

In [7]:
SPLIT_POLICY=ContinuousRegressionSplitPolicy(evaluation_mode="shuffled_random_snapshot",purpose="educational_benchmark",train_fraction=float(sc["train_fraction"]),validation_fraction=float(sc["validation_fraction"]),test_fraction=float(sc["test_fraction"]),random_seed=int(sc["random_seed"]),shuffle=bool(sc["shuffle_random_split"]),stratify_by=None,educational_justification="Static educational snapshot; no chronological field or artificial target bins.",operational_validity="unconfirmed",temporal_contract_status="resolved_static_snapshot",feature_inference_availability="unconfirmed")
assert sc["stratification_field"] is None and tuple(sc["identifier_grouping"])==IDENTIFIER_COLUMNS

## 8. Deterministic Non-Stratified Regression Partitioning

Two sklearn stages split source positions using seeds 42 and 43 and stratify=None. Target values, bins, diagnostics, and membership hashes never drive assignment.

In [8]:
partitions=split_continuous_regression_dataset(prepared_df,policy=SPLIT_POLICY,identifier_columns=IDENTIFIER_COLUMNS)
train_df,validation_df,test_df=partitions.train,partitions.validation,partitions.test
assert (len(train_df),len(validation_df),len(test_df))==(721,154,155)
display(pd.DataFrame([{"partition":k,"rows":len(v)} for k,v in partitions.as_mapping().items()]))

    partition  rows
0       train   721
1  validation   154
2        test   155


## 9. Partition Integrity, Target Diagnostics, and Repeated Profiles

Diagnostics never gate or revise the first valid split. Technical row-occurrence membership is post-assignment integrity evidence only.

In [9]:
partition_validation=validate_regression_partitions(prepared_df,partitions,identifier_columns=IDENTIFIER_COLUMNS,target_column=TARGET_COLUMN)
repeat=split_continuous_regression_dataset(prepared_df,policy=SPLIT_POLICY,identifier_columns=IDENTIFIER_COLUMNS); assert partitions.membership_mapping()==repeat.membership_mapping()
TARGET_DIAGNOSTICS=dict(partition_validation.target_diagnostics)
repeated_profile_evidence=analyze_repeated_profiles_across_partitions(prepared_df,partitions,feature_columns=FEATURE_COLUMNS,target_column=TARGET_COLUMN,identifier_columns=IDENTIFIER_COLUMNS)
assert repeated_profile_evidence["proven_duplicate_identity"] is False and repeated_profile_evidence["source_exact_row_equality_group_count"]==11 and repeated_profile_evidence["source_repeated_feature_profile_group_count"]==19 and repeated_profile_evidence["target_conflicting_feature_profile_group_count"]==9
display(pd.DataFrame(TARGET_DIAGNOSTICS).T)

           count minimum maximum       mean median standard_deviation  \
train        721    2.33    82.6  36.152732  35.08          16.803538   
validation   154    4.83    76.8  34.727208  34.53          15.917696   
test         155    4.57   81.75  35.344452  33.02           17.05833   

                                                    quantiles diagnostic_only  \
train       {'1%': 7.054, '5%': 11.36, '25%': 23.89, '50%'...            True   
validation  {'1%': 6.8906, '5%': 11.326, '25%': 22.51, '50...            True   
test        {'1%': 6.841600000000001, '5%': 10.613, '25%':...            True   

           used_for_assignment_or_seed_selection  
train                                      False  
validation                                 False  
test                                       False  


## 10. Frozen Preprocessing Contract

No learned preprocessing is fitted here. Model-specific learning is deferred to Notebook 03 and must occur inside training data or folds.

In [10]:
PREPROCESSING_CONTRACT={"input_feature_type":"numerical_only","categorical_strategy":"not_applicable","numerical_scaling":"model_specific","learned_fit_scope":"inside_training_data_or_training_fold_only","learned_transformations_fitted_in_notebook_02":False,"persisted_target":"continuous numeric value on original MPa scale","target_encoding":"not_applicable","deferred_to_notebook_03":["regression baseline comparison","cross-validation","candidate model selection","hyperparameter selection","model-specific scaling","feature ablation/redundancy evaluation","nonlinear model evaluation","interaction-capable model evaluation","repeated-profile sensitivity analysis","target-extreme error sensitivity","regression residual/error analysis"],"prohibited_in_notebook_02":["model fit","model score","cross-validation","candidate comparison","hyperparameter selection","feature selection","target transformation selection","test predictive metric calculation","residual calculation","test-set decision use"]}

## 11. Preparation Artifact Contracts

Manifests freeze source lineage, continuous roles, non-stratified membership, diagnostic evidence, preservation, and readiness.

In [11]:
SPLIT_ID="shuffled-70-15-15-seed-42"; PREPARED_PATH=Path("data/processed")/DATASET_SLUG/"prepared.csv"; SPLIT_ROOT=Path("data/processed")/DATASET_SLUG/"splits"/SPLIT_ID
PARTITION_PATHS={n:SPLIT_ROOT/f"{n}.csv" for n in ("train","validation","test")}; ARTIFACT_ROOT=Path("artifacts/preparation")/DATASET_SLUG
MANIFEST_PATHS={"preparation_manifest":ARTIFACT_ROOT/"preparation-manifest.json","feature_manifest":ARTIFACT_ROOT/"feature-manifest.json","split_manifest":ARTIFACT_ROOT/"split-manifest.json","quality_evidence":ARTIFACT_ROOT/"quality-evidence.json"}; PREPARATION_HANDOFF_PATH=ARTIFACT_ROOT/"preparation-handoff.json"
PREPARED_SHA256=fingerprint_dataframe_csv(prepared_df); PARTITION_SHA256={n:fingerprint_dataframe_csv(f) for n,f in partitions.as_mapping().items()}
UPSTREAM_LINEAGE={"path":EXPLORATION_HANDOFF_PATH.as_posix(),"schema_version":h["schema_version"],"dataset_slug":DATASET_SLUG,"sha256":EXPLORATION_SHA256,"source_dataset_id":UCI_DATASET_ID,"source_dataset_sha256":SOURCE_SHA256}
SOURCE_TYPE_RESOLUTIONS=[{"column":"Blast Furnace Slag","source_declared_type":"Integer","observed_dtype":str(raw_df["Blast Furnace Slag"].dtype),"effective_analytical_type":"Continuous numeric","decimal_row_count":DECIMAL_SLAG_ROWS,"operation":"Preserve released decimal values exactly; do not round, truncate, or coerce them to integer.","values_changed":0,"rows_removed":0}]
READINESS={"notebook_01_handoff_validated":True,"source_independently_revalidated":True,"prepared_projection_materialized":True,"split_materialized":True,"partition_integrity_validated":True,"preparation_handoff_reloadable":True,"educational_model_selection_ready":True,"test_partition_sealed":True,"test_partition_evaluated":False,"model_selected":False,"final_model_trained":False,"operational_modeling_ready":False,"operational_validity":"unconfirmed","temporal_contract_status":"resolved_static_snapshot","feature_inference_availability":"unconfirmed"}

In [12]:
existing_preparation_manifest_path=PROJECT.root/MANIFEST_PATHS["preparation_manifest"]
RUNTIME_VERSION_EVIDENCE=None
if existing_preparation_manifest_path.is_file():
    existing_preparation_manifest=json.loads(existing_preparation_manifest_path.read_text(encoding="utf-8"))
    candidate_runtime_versions=existing_preparation_manifest.get("runtime_versions")
    if isinstance(candidate_runtime_versions,dict):
        RUNTIME_VERSION_EVIDENCE=candidate_runtime_versions
preparation_manifest=build_preparation_manifest(dataset_slug=DATASET_SLUG,source_path=source_identity.source_path,source_sha256=SOURCE_SHA256,prepared_path=PREPARED_PATH,prepared_sha256=PREPARED_SHA256,raw_report=raw_validation,prepared_report=prepared_validation,preparation=prepared_result,raw_fingerprint_before=RAW_LOGICAL_FINGERPRINT,raw_fingerprint_after=fingerprint_dataframe(raw_df),source_sha256_after=fingerprint_file(DATASET_FILE),deterministic_rules=[],readiness=READINESS,source_identity=source_identity.as_dict(),upstream_exploration=UPSTREAM_LINEAGE,source_type_resolutions=SOURCE_TYPE_RESOLUTIONS,runtime_version_evidence=RUNTIME_VERSION_EVIDENCE)
feature_manifest=build_feature_manifest(dataset_slug=DATASET_SLUG,identifier_columns=IDENTIFIER_COLUMNS,feature_columns=FEATURE_COLUMNS,numerical_features=NUMERICAL_FEATURES,categorical_features=CATEGORICAL_FEATURES,categorical_expected_values={},target_column=TARGET_COLUMN,target_classes=(),expected_dtypes={"raw":EXPECTED_DTYPES,"prepared":EXPECTED_DTYPES},preprocessing_contract=PREPROCESSING_CONTRACT,prohibited_predictors=(TARGET_COLUMN,"target-derived values","technical membership tokens"),problem_type="continuous_regression",target_semantics=pc["target_semantics"],target_unit=pc["target_unit"],prediction_output=pc["prediction_output"])
split_manifest=build_split_manifest(dataset_slug=DATASET_SLUG,policy=SPLIT_POLICY,partitions=partitions,validation=partition_validation,partition_paths=PARTITION_PATHS,partition_sha256=PARTITION_SHA256,repeated_profile_evidence=repeated_profile_evidence)
quality_evidence=build_quality_evidence(dataset_slug=DATASET_SLUG,raw_report=raw_validation,prepared_report=prepared_validation,partition_report=partition_validation,preparation=prepared_result,fingerprints={"source_sha256_before":SOURCE_SHA256,"source_sha256_after":fingerprint_file(DATASET_FILE),"raw_logical_fingerprint_before":RAW_LOGICAL_FINGERPRINT,"raw_logical_fingerprint_after":fingerprint_dataframe(raw_df),"prepared_sha256":PREPARED_SHA256,"partition_sha256":PARTITION_SHA256},readiness=READINESS,preservation_checks={"raw_dataframe_unchanged":True,"source_file_unchanged":True,"rows_removed":0,"values_changed":0,"features_removed":0,"target_values_changed":0,"duplicate_rows_removed":0,"generic_outlier_treatment_applied":False,"learned_preprocessing_fitted":False,"target_transformed":False,"blast_furnace_slag_decimals_preserved":True},repeated_profile_evidence=repeated_profile_evidence,source_type_resolutions=SOURCE_TYPE_RESOLUTIONS)
COMPONENT_PAYLOADS={"preparation_manifest":preparation_manifest,"feature_manifest":feature_manifest,"split_manifest":split_manifest,"quality_evidence":quality_evidence}
preparation_handoff_manifest=build_preparation_handoff_manifest(dataset_slug=DATASET_SLUG,component_paths=MANIFEST_PATHS,component_payloads=COMPONENT_PAYLOADS,readiness=READINESS,upstream_exploration=UPSTREAM_LINEAGE)

## 12. Atomic Artifact Materialization

A staged transaction uses overwrite=False: equivalent reruns are reused and divergent artifacts fail closed.

In [13]:
write_result=write_preparation_artifacts(project_root=PROJECT.root,csv_artifacts={PREPARED_PATH:prepared_df,**{PARTITION_PATHS[n]:f for n,f in partitions.as_mapping().items()}},json_artifacts={**{MANIFEST_PATHS[n]:p for n,p in COMPONENT_PAYLOADS.items()},PREPARATION_HANDOFF_PATH:preparation_handoff_manifest},overwrite=False)
display(pd.DataFrame([{"path":p,"status":s,"sha256":dict(write_result.sha256)[p]} for p,s in write_result.statuses]))

                                                path             status  \
0  artifacts/preparation/concrete-compressive-str...  reused_equivalent   
1  artifacts/preparation/concrete-compressive-str...  reused_equivalent   
2  artifacts/preparation/concrete-compressive-str...  reused_equivalent   
3  artifacts/preparation/concrete-compressive-str...  reused_equivalent   
4  artifacts/preparation/concrete-compressive-str...  reused_equivalent   
5  data/processed/concrete-compressive-strength/p...  reused_equivalent   
6  data/processed/concrete-compressive-strength/s...  reused_equivalent   
7  data/processed/concrete-compressive-strength/s...  reused_equivalent   
8  data/processed/concrete-compressive-strength/s...  reused_equivalent   

                                              sha256  
0  e0a85720a3badc03c722cdd298dee42963180a3962174b...  
1  043be5e24d5cf77c1c4716d2441c7bd93a8d5db8e8ee6b...  
2  0bb7a3e46cc1dca21dac565f670d9915e541f0f0632651...  
3  7d1834ae5d4f97885e5854a8cc

## 13. Preparation Readiness

Educational model selection is ready through the frozen handoff. Test is sealed and unevaluated; no model or operational claim exists.

In [14]:
display(pd.DataFrame([READINESS]))

   notebook_01_handoff_validated  source_independently_revalidated  \
0                           True                              True   

   prepared_projection_materialized  split_materialized  \
0                              True                True   

   partition_integrity_validated  preparation_handoff_reloadable  \
0                           True                            True   

   educational_model_selection_ready  test_partition_sealed  \
0                               True                   True   

   test_partition_evaluated  model_selected  final_model_trained  \
0                     False           False                False   

   operational_modeling_ready operational_validity  temporal_contract_status  \
0                       False          unconfirmed  resolved_static_snapshot   

  feature_inference_availability  
0                    unconfirmed  


## 14. Fresh Handoff Reload and Lineage Validation

Reload authenticates upstream/component/CSV hashes and reconstructs all data without resplitting.

In [15]:
reloaded_handoff=load_and_validate_preparation_handoff(project_root=PROJECT.root,preparation_handoff_path=PREPARATION_HANDOFF_PATH); manifests=reloaded_handoff.manifests
assert len(reloaded_handoff.prepared)==1030 and (len(reloaded_handoff.train),len(reloaded_handoff.validation),len(reloaded_handoff.test))==(721,154,155)
assert tuple(manifests["feature_manifest"]["feature_columns"])==FEATURE_COLUMNS and manifests["feature_manifest"]["problem_type"]=="continuous_regression"
assert manifests["split_manifest"]["stratification"] is None and manifests["quality_evidence"]["readiness"]["test_partition_sealed"] is True and manifests["quality_evidence"]["readiness"]["test_partition_evaluated"] is False
print("Preparation handoff reload: PASSED")

Preparation handoff reload: PASSED
